In [1]:
import tokenizers
from pathlib import Path
from datasets import load_dataset, concatenate_datasets, interleave_datasets
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
import json

/home/rod/projects/minilm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
cosmopedia = load_dataset("parquet", data_files="data/smollm-corpus-dev/cosmopedia-v2/*.parquet", split="train", streaming=True)
fineweb = load_dataset("parquet", data_files="data/smollm-corpus-dev/fineweb-edu-dedup/*.parquet", split="train", streaming=True)
# python = load_dataset("parquet", data_files="data/smollm-corpus-dev/python-edu/*.parquet", split="train", streaming=True)
combined = interleave_datasets([cosmopedia, fineweb])

In [3]:
# x = next(iter(cosmopedia))
# print(json.dumps({k: v[:50] if isinstance (v, str) else v for k, v in x.items()}, indent=1))

In [4]:
# print(json.dumps({k: str(v)[:50] for k, v in next(iter(fineweb)).items()}, indent=1))
# print(json.dumps({k: v[:50] if isinstance (v, str) else v for k, v in next(iter(python)).items()}, indent=1))

In [5]:
# next(iter(fineweb))

In [7]:
def batch_iterator(dataset, text_column="text", batch_size=1000):
    batch = []

    for example in dataset:
        text = example.get(text_column)

        batch.append(text)

        if len(batch) == batch_size:
            yield batch
            batch = []

    if batch:
        yield batch

In [10]:
tokenizer = Tokenizer(models.BPE())
tokenizer.normalizer = normalizers.NFKC()
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

VOCAB_SIZE = 16_000

special_tokens = [
    "<|pad|>",
    "<|eos|>",

    "<|system|>",
    "<|user|>",
    "<|assistant|>",
    "<|tool|>",

    "<|end_message|>",

    "<|think|>",
    "<|end_think|>",

    "<|tool_call|>",
    "<|end_tool_call|>",
]

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=special_tokens,
)

tokenizer.train_from_iterator(batch_iterator(combined), trainer=trainer)
# tokenizer.save("tokenizer.json")

In [21]:
text = "The quick brown fox jumps over the lazy dog"

tokenizer.decoder = tokenizers.decoders.ByteLevel()
encoding = tokenizer.encode(text)
ids = encoding.ids

print([tokenizer.decode([i]) for i in ids])

# decoded = tokenizer.decode(ids)

# print(decoded)

['The', ' quick', ' brown', ' f', 'ox', ' j', 'umps', ' over', ' the', ' l', 'az', 'y', ' dog']
